In [26]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [27]:
data_gdynia_typed = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

data_gdynia_raw = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    parse_dates=["date_of_checking"],
)

data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_categories.head()


,addr_street,function_clean,category
id,,,
219,Stefana Żeromskiego,administracja,organizacje publiczne
258,3 Maja,administracja,organizacje publiczne
374,Starowiejska,administracja,organizacje publiczne
495,10 Lutego,administracja,organizacje publiczne
86,Jana z Kolna,administracja,organizacje publiczne


1.

In [28]:
types = pd.concat(
    {
        "raw": data_gdynia_raw.dtypes,
        "typed": data_gdynia_typed.dtypes,
        "is_empty": data_gdynia_typed.isna().all(),
    },
    axis=1,
)

In [29]:
diff_types = types[(types["raw"] != types["typed"]) & ~types["is_empty"]]
empty = types[types["is_empty"]]


In [30]:
assert len(diff_types) == 5
assert (empty["raw"] == "float64").all()

2.

In [31]:
raw_na = data_gdynia_raw["confidence"].where(
    data_gdynia_raw["arch_style"] != "modernizm"
)
typed_na = data_gdynia_typed["confidence"].where(
    data_gdynia_typed["arch_style"] != "modernizm"
)

In [32]:
print(f"dtype: raw wo Nan: {data_gdynia_raw['confidence'].dtype}")
print(f"dtype: raw with Nan: {raw_na.dtype}")
print(f"dtype: typed wo Nan: {data_gdynia_typed['confidence'].dtype}")
print(f"dtype: typed with Nan: {typed_na.dtype}")
print(f"Nan value raw: {raw_na[raw_na.isna()].head(1)}")
print(f"Nan value typed: {typed_na[typed_na.isna()].head(1)}")

dtype: raw wo Nan: int64
dtype: raw with Nan: float64
dtype: typed wo Nan: Int64
dtype: typed with Nan: Int64
Nan value raw: id
9   NaN
Name: confidence, dtype: float64
Nan value typed: id
9    <NA>
Name: confidence, dtype: Int64


In [33]:
assert raw_na.isna().sum() == typed_na.isna().sum() == 466
assert raw_na.mean() == typed_na.mean()
assert np.isclose(raw_na.mean(), 2.4574, atol=0.001)
assert raw_na.sum() == typed_na.sum() == 1268

3.

In [34]:
atrybutes = (
    data_gdynia_typed.loc[
        :, ["osm_id", "addr_street", "source", "number_of_storeys", "date_of_checking"]
    ]
    .agg(["size", "count", "nunique", "hasnans", "is_unique"])
    .T
)
atrybutes

,size,count,nunique,hasnans,is_unique
osm_id,982,982,982,False,True
addr_street,982,713,44,True,False
source,982,982,1,False,False
number_of_storeys,982,982,15,False,False
date_of_checking,982,982,7,False,False


In [35]:
assert atrybutes.loc["osm_id", "hasnans"] == data_gdynia_typed["osm_id"].hasnans
assert atrybutes.loc["osm_id", "is_unique"] == data_gdynia_typed["osm_id"].is_unique

assert (atrybutes.loc["addr_street"] == [982, 713, 44, True, False]).all()

4.

In [36]:
data_categories_styles = data_gdynia_typed.assign(
    arch_style=lambda d: d["arch_style"].astype("category")
)


In [37]:
data_categories_styles["arch_style"].cat.categories

Index(['klasycyzm', 'modernizm', 'postmodernizm', 'tradycjonalizm',
       'współczesność'],
      dtype='str')

In [38]:
data_gdynia_typed.loc[
    data_gdynia_typed["addr_street"] == "Świętojańska", "arch_style"
].value_counts()

arch_style
modernizm        90
klasycyzm         9
współczesność     7
Name: count, dtype: int64

In [39]:
data_categories_Sw = data_categories_styles.loc[
    data_categories_styles["addr_street"] == "Świętojańska", "arch_style"
].value_counts()
data_categories_Sw

arch_style
modernizm         90
klasycyzm          9
współczesność      7
postmodernizm      0
tradycjonalizm     0
Name: count, dtype: int64

In [40]:
assert (data_categories_Sw == [90, 9, 7, 0, 0]).all()

In [41]:
data_gdynia_long_categories["category"].value_counts()

category
handel                       289
garaże                       216
gastronomia                  178
usługi                       153
mieszkaniowa                 101
zdrowie                       93
przemysł i infrastruktura     57
organizacje publiczne         53
pustostan                     51
finanse                       46
kultura i rozrywka            34
biura                         28
brak parteru                  19
noclegi                       15
transport                     10
kult religijny                 7
Name: count, dtype: int64

5.

In [42]:
sum_vacant = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] == "pustostan"
    ]
    .groupby("addr_street")
    .size()
)
len(sum_vacant)

17

In [43]:
sum_functions = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] != "brak parteru"
    ]
    .groupby("addr_street")
    .size()
)
len(sum_functions)

44

In [44]:
pd.testing.assert_series_equal(
    sum_vacant.div(sum_functions), sum_vacant / sum_functions
)

In [ ]:
sum_vacant.div(sum_functions, fill_value=0).rename("share_vacant")

addr_street
10 Lutego                                0.056604
3 Maja                                   0.034483
Aleja Jana Pawła II                      0.000000
Aleja Marszałka Józefa Piłsudskiego      0.000000
Antoniego Abrahama                       0.042735
Antoniego Hryniewickiego                 0.115385
Arkadiusza Rybickiego                    0.250000
Armii Krajowej                           0.030303
Dworcowa                                 0.000000
Franciszka Żwirki i Stanisława Wigury    0.052632
Generała Józefa Bema                     0.071429
Hieronima Derdowskiego                   0.000000
Hilarego Jastaka                         0.000000
Jana Kilińskiego                         0.000000
Jana z Kolna                             0.192308
Janka Wiśniewskiego                      0.000000
Jerzego Waszyngtona                      0.038462
Juliusza Słowackiego                     0.000000
Józefa Wybickiego                        0.000000
Kazimierza Pułaskiego                 

6.

In [ ]:
share_vacant = (
    data_gdynia_long_categories.loc[
        data_gdynia_long_categories["category"] != "brak parteru"
    ]
    .assign(is_vacant=lambda d: d["category"] == "pustostan")
    .groupby("addr_street")
    .agg(
        num_premises=("addr_street", "size"),
        num_vacant=("is_vacant", "sum"),
        share_vacant=("is_vacant", "mean"),
    )
)


In [ ]:
pd.testing.assert_series_equal(
    sum_vacant.div(sum_functions, fill_value=0).rename("share_vacant"),
    share_vacant["share_vacant"],
)